# 📄 Invoice Data Extraction System

**Architecture:** Open-Source VLM (Qwen2-VL) + PaddleOCR + Grounding + Validation

This notebook demonstrates a production-grade invoice extraction pipeline that:
1. Uses an **open-source VLM** (Qwen2-VL-2B) — no API keys, no cloud dependency
2. Uses **PaddleOCR** (deep learning based) for pixel-accurate word detection
3. **Grounds** VLM outputs to OCR bounding boxes to prevent hallucinations
4. **Validates** extracted data (arithmetic checks, date formats, currency codes)
5. Produces **composite confidence scores** for every field

---
⚙️ **Setup:** Enable GPU T4 in Settings → Accelerator → GPU T4 x2

## Step 1: Install Dependencies

In [ ]:
%%capture
!pip install -q 'paddlepaddle-gpu>=2.6.0,<3.0.0' 'paddleocr>=2.7.0,<3.0.0' pdf2image rapidfuzz pydantic pydantic-settings Pillow reportlab python-dotenv opencv-python-headless
!pip install -q transformers accelerate torch
!apt-get install -y poppler-utils -qq
print('✅ All dependencies installed')

## Step 2: Clone Project from GitHub

In [ ]:
import subprocess, sys, os
repo_dir = '/kaggle/working/invoice'
if os.path.exists(repo_dir):
    subprocess.run(['git', '-C', repo_dir, 'pull'], check=True)
    print('✅ Project updated (git pull)')
else:
    subprocess.run(['git', 'clone', 'https://github.com/singh-anjali24/Invoice.git', repo_dir], check=True)
    print('✅ Project cloned successfully')
sys.path.insert(0, repo_dir)
os.chdir(repo_dir)

## Step 3: Generate Sample Invoices
Generate 3 diverse sample invoices (US/USD, India/INR, EU/EUR) for testing.

In [ ]:
!python scripts/generate_samples.py

## Step 4: Load the Open-Source VLM on GPU
Loading **Qwen2-VL-2B-Instruct** — a fully open-source vision-language model.

This takes ~1 minute on first run (downloading model weights). Subsequent runs use cached weights.

In [ ]:
from transformers import Qwen2VLForConditionalGeneration, AutoProcessor
import torch

MODEL_NAME = 'Qwen/Qwen2-VL-2B-Instruct'

print('⏳ Loading open-source VLM...')
processor = AutoProcessor.from_pretrained(MODEL_NAME, trust_remote_code=True)
model = Qwen2VLForConditionalGeneration.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16,
    device_map='auto',
    trust_remote_code=True
)
print(f'✅ Model loaded on: {model.device}')
print(f'   Model: {MODEL_NAME}')
print(f'   Parameters: 2B')
print(f'   License: Apache 2.0 (fully open-source)')

## Step 5: Define the VLM Extraction Function
This function sends an invoice image to the **local** VLM (running on this GPU) and gets structured JSON output.

In [ ]:
from PIL import Image
from pdf2image import convert_from_path
import json, re, time
import numpy as np

EXTRACTION_PROMPT = """You are an expert invoice data extraction system. Given this scanned invoice image, extract ALL fields into valid JSON.

Return ONLY this exact JSON structure (no markdown, no extra text):
{
  "invoice_number": "string",
  "invoice_date": "YYYY-MM-DD",
  "due_date": "YYYY-MM-DD or null",
  "vendor_name": "string",
  "vendor_tax_id": "string or null",
  "buyer_name": "string or null",
  "line_items": [{"description": "string", "quantity": 1, "unit_price": 0.0, "line_total": 0.0}],
  "total_amount": 0.0,
  "currency": "USD"
}

RULES:
1. Dates MUST be ISO 8601: YYYY-MM-DD
2. If a field is NOT visible, set it to null
3. Currency: infer from symbols (₹=INR, $=USD, €=EUR)
4. total_amount = FINAL payable total (after tax)
5. vendor_name = company that ISSUED the invoice
6. Return ONLY valid JSON, nothing else"""


def extract_with_local_vlm(pdf_path):
    """Extract invoice data using the LOCAL open-source VLM (no API calls)."""
    # Convert PDF to image
    images = convert_from_path(pdf_path, dpi=300)
    pil_image = images[0]
    pil_image.thumbnail((1024, 1024))  # Resize to fit GPU memory
    
    # Prepare input for the model
    messages = [{'role': 'user', 'content': [
        {'type': 'image', 'image': pil_image},
        {'type': 'text', 'text': EXTRACTION_PROMPT}
    ]}]
    
    text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = processor(text=[text], images=[pil_image], return_tensors='pt').to(model.device)
    
    # Generate response LOCALLY on the GPU
    start = time.time()
    with torch.no_grad():
        output_ids = model.generate(**inputs, max_new_tokens=2048, temperature=0.1, do_sample=True)
    elapsed = time.time() - start
    
    # Decode the output
    result_text = processor.batch_decode(
        output_ids[:, inputs.input_ids.shape[1]:], skip_special_tokens=True
    )[0]
    
    # Parse JSON from the response
    json_match = re.search(r'\{.*\}', result_text, re.DOTALL)
    if json_match:
        parsed = json.loads(json_match.group())
        return parsed, np.array(pil_image.convert('RGB'))[:, :, ::-1], elapsed  # Return as BGR for OCR
    
    raise ValueError(f'VLM did not return valid JSON: {result_text[:200]}')


print('✅ Extraction function defined')

## Step 6: Extract from US Invoice (ACME Technologies)
This calls the **local** open-source VLM — no internet, no API key.

In [ ]:
pdf_path = 'samples/invoice_us_acme.pdf'
print(f'📄 Processing: {pdf_path}')
print('⏳ Running LOCAL VLM extraction (no API call)...')

vlm_result, image_bgr, vlm_time = extract_with_local_vlm(pdf_path)

print(f'\n✅ VLM extraction complete in {vlm_time:.1f}s')
print(f'\n--- VLM Raw Output ---')
print(json.dumps(vlm_result, indent=2))

## Step 7: Run PaddleOCR + Grounding + Validation Pipeline
Now we run the **same pipeline** that runs in Docker:
1. **PaddleOCR** → Detects every word with pixel-accurate bounding boxes
2. **Grounding** → Matches VLM outputs to OCR words (prevents hallucinations)
3. **Validation** → Arithmetic checks, date format verification
4. **Confidence Scoring** → Composite score per field

In [ ]:
from src.ocr import get_word_map
from src.grounding import ground_all_fields
from src.validation import validate_extraction
from src.confidence import compute_confidences
from src.schemas import RawInvoiceExtraction, ExtractionMetadata

# Step 7a: Run PaddleOCR
print('📝 Running PaddleOCR...')
word_map = get_word_map(image_bgr, page=0)
print(f'   Detected {len(word_map)} words with bounding boxes')

# Step 7b: Ground VLM outputs to OCR bounding boxes
print('\n📍 Grounding VLM outputs to OCR coordinates...')
raw = RawInvoiceExtraction(**vlm_result)
invoice = ground_all_fields(raw, word_map)

# Count grounded fields
grounded = sum(1 for f in [
    invoice.invoice_number, invoice.invoice_date, invoice.due_date,
    invoice.vendor_name, invoice.vendor_tax_id, invoice.buyer_name,
    invoice.total_amount, invoice.currency
] if f.bounding_box is not None)
print(f'   Grounded {grounded}/8 top-level fields to bounding boxes')

# Step 7c: Validate
print('\n✅ Running validation checks...')
validation = validate_extraction(invoice)
print(f'   Validation score: {validation.overall_score:.2f}')
print(f'   Warnings: {len(validation.warnings)}')
for w in validation.warnings:
    print(f'   ⚠️  {w}')

# Step 7d: Compute confidence scores
print('\n📊 Computing confidence scores...')
invoice = compute_confidences(invoice, validation, word_map)

# Attach metadata
h, w_img = image_bgr.shape[:2]
invoice.metadata = ExtractionMetadata(
    source_file='invoice_us_acme.pdf',
    model_used='Qwen2-VL-2B (open-source)',
    processing_time_seconds=round(vlm_time, 2),
    image_width=w_img,
    image_height=h,
    ocr_words_detected=len(word_map),
    validation_warnings=validation.warnings,
)

## Step 8: Final Structured JSON Output
This is the complete extraction result — with confidence scores and bounding boxes for every field.

In [ ]:
# Full output (with metadata)
output = json.loads(invoice.model_dump_json())
print('=' * 60)
print('FINAL EXTRACTION RESULT')
print('=' * 60)
print(json.dumps(output, indent=2))

# Summary
print(f'\n📊 Summary:')
print(f'   Invoice:  {invoice.invoice_number.value} (confidence: {invoice.invoice_number.confidence:.2f})')
print(f'   Vendor:   {invoice.vendor_name.value} (confidence: {invoice.vendor_name.confidence:.2f})')
print(f'   Total:    {invoice.currency.value} {invoice.total_amount.value} (confidence: {invoice.total_amount.confidence:.2f})')
print(f'   Items:    {len(invoice.line_items)}')
print(f'   Model:    Qwen2-VL-2B (open-source, local GPU)')
print(f'   OCR:      PaddleOCR (deep learning based)')

## Step 9: Visual Verification
Display the invoice image alongside the extracted data so you can visually verify accuracy.

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(10, 14))
plt.imshow(image_bgr[:, :, ::-1])
plt.axis('off')
plt.title('Invoice Image — Compare with JSON above', fontsize=14)
plt.show()

## Step 10: Flat / Downstream JSON Format
The exact shape a downstream accounting system would consume.

In [ ]:
downstream = invoice.to_downstream_json()
print('=' * 60)
print('DOWNSTREAM FORMAT (flat)')
print('=' * 60)
print(json.dumps(downstream, indent=2))

## Step 11: Process All 3 Invoices (US, India, EU)
Demonstrating the system works across different countries, currencies, and formats.

In [ ]:
invoices = [
    ('🇺🇸 US Invoice (USD)',    'samples/invoice_us_acme.pdf'),
    ('🇮🇳 India Invoice (INR)', 'samples/invoice_india_tcs.pdf'),
    ('🇪🇺 EU Invoice (EUR)',    'samples/invoice_eu_mueller.pdf'),
]

print('=' * 70)
print('MULTI-FORMAT EXTRACTION RESULTS')
print('=' * 70)

for name, path in invoices:
    print(f'\n{name}')
    print('-' * 50)
    
    # Extract with local VLM
    vlm_result, img, vlm_time = extract_with_local_vlm(path)
    
    # Run full pipeline
    word_map = get_word_map(img, page=0)
    raw = RawInvoiceExtraction(**vlm_result)
    invoice = ground_all_fields(raw, word_map)
    validation = validate_extraction(invoice)
    invoice = compute_confidences(invoice, validation, word_map)
    
    grounded = sum(1 for f in [
        invoice.invoice_number, invoice.invoice_date, invoice.due_date,
        invoice.vendor_name, invoice.vendor_tax_id, invoice.buyer_name,
        invoice.total_amount, invoice.currency
    ] if f.bounding_box is not None)
    
    print(f'  Invoice #:     {invoice.invoice_number.value}')
    print(f'  Vendor:        {invoice.vendor_name.value}')
    print(f'  Total:         {invoice.currency.value} {invoice.total_amount.value}')
    print(f'  Line Items:    {len(invoice.line_items)}')
    print(f'  Grounded:      {grounded}/8 fields')
    print(f'  Validation:    {validation.overall_score:.2f} ({len(validation.warnings)} warnings)')
    print(f'  VLM Time:      {vlm_time:.1f}s')
    print(f'  Model:         Qwen2-VL-2B (open-source)')
    print(f'  OCR:           PaddleOCR')

## ✅ Summary

| Component | Technology | Why |
|---|---|---|
| **VLM** | Qwen2-VL-2B (open-source) | No API dependency, runs on free GPU, full data privacy |
| **OCR** | PaddleOCR (deep learning) | Superior accuracy vs Tesseract, pure Python, no system packages |
| **Grounding** | RapidFuzz + sliding window | Prevents VLM hallucinations by anchoring to OCR text |
| **Validation** | Custom rules engine | Arithmetic checks, date/currency verification |
| **Scoring** | Weighted composite | 3-signal confidence: OCR + Grounding + Validation |

**Key Design Decision:** The VLM is a swappable plug-in. This same pipeline works with Gemini (cloud), Qwen (local), or any future model — only the VLM module changes.

**Scaling Note:** In production with more GPU memory, simply swap `Qwen2-VL-2B` for `Qwen2-VL-7B` or `Qwen2.5-VL-7B` for even higher accuracy. The pipeline code remains identical.